## Чтение данных и создание итоговой таблицы

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

sheets = pd.ExcelFile('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx')
print(sheets.sheet_names)

data=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table1 (basic)',skiprows=[0])
data.drop(['QualityRequirement','QualNo','CustID','CustVer','InternalVer','MetalRavneQualityName','SteelGroup','Month','Year'],axis=1,inplace=True)
data.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data.dropna(subset=['Cr_Last_EOP','Mo_Last_EOP','Ni_Last_EOP','V_Last_EOP','W_Last_EOP'],inplace=True)


data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table2 (limits)',skiprows=[0])
data_.drop(['QualityRequirement','QualNo','CustID','CustVer','InternalVer','MetalRavneQualityName','SteelGroup','Month','Year'],axis=1,inplace=True)
data_.drop(['Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1,inplace=True)
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table4 (alloys)',skiprows=[0])
data.drop(['LFVD_FeCrA', 'LFVD_FeCrC','LFVD_NiGran', 'LFVD_NiKatode','LFVD_FeMo', 'LFVD_Polymox','LFVD_FeV','LFVD_FeW72','LFVD_WPaketi'],axis=1,inplace=True)
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table8 (scrap)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table9 (all. scr.)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table10 (unall. scr.)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

display(data.head(3))
print('всего строк=',len(data))
print('всего столбцов=',len(data.columns))
data.to_csv('итоговая таблица.csv',sep=';',index=False)

### Удаление данных в которых дисперсия равна нулю

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
data=pd.read_csv('итоговая таблица.csv',sep=';')
#LFVD_CaS LFVD_CoMet  LFVD_Cu LFVD_FeB	LFVD_FeBŽica	LFVD_FeCrNit  LFVD_FeMnA  LFVD_FeNbTa	LFVD_FeNbTafi13 LFVD_FeSiZrŽica	LFVD_FeTi	LFVD_FeTifi13
#LFVD_NiKatode	LFVD_NiOksid	LFVD_OdpCu  LFVD_SiMet  LFVD_KarboritZaVpih	LFVD_Ni90 

display(data.describe())
del_=[]
for i in data.describe().columns:
    if data.describe().loc['std',i]==0:
        del_.append(i)

del_.remove('Ni_LowerLimit')
del_.remove('Ni_Target')
del_.remove('W_LowerLimit')
del_.remove('W_Target')

data.drop(del_,axis=1,inplace=True)
data.drop(['Fill#1_ScrapName','Fill#1_ScrapWeight','Fill#2_ScrapName','Fill#2_ScrapWeight','Fill#3_ScrapName','Fill#3_ScrapWeight','Fill#4_ScrapName',
          'Fill#4_ScrapWeight','Fill#5_ScrapName','Fill#5_ScrapWeight','Fill#6_ScrapName','Fill#6_ScrapWeight'],axis=1,inplace=True)

print('всего строк=',len(data))
print('всего столбцов=',len(data.columns))
data.to_csv('итоговая таблица_испр.csv',sep=';',index=False)

### Проверка взаимных корреляций

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data_=data.drop(['HeatNo','Date'],axis=1)

corr=data_.corr()

print('число столбцов в таблице=',len(data_.corr()))

display(corr.head(3))
gran_corr_value=0.8
corr_={i:[] for i in corr.columns}
for i in range(len(corr.columns)):    
    mas=corr[corr.columns[i]].tolist()
    el=mas[i]
    ind_max=[j for j in range(len(mas)) if (mas[j]>gran_corr_value)&(j!=i)]
    mas.remove(mas[i])
    corr_[corr.columns[i]]=[min(mas),max(mas)]+[corr.columns[j] for j in ind_max]
    print(corr.columns[i],':',corr_[corr.columns[i]])



### Гистограмма превышения целевого значения по компонентам

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data['Cr_exceed']=data.apply(lambda x:x['Cr_Final']-x['Cr_Target'],axis=1)
data['Ni_exceed']=data.apply(lambda x:x['Ni_Final']-x['Ni_Target'],axis=1)

print('в среднем разница между Cr_Target и Cr_LowerLimit =',data.apply(lambda x:x['Cr_Target']-x['Cr_LowerLimit'],axis=1).mean())
print('в среднем разница между Ni_Target и Ni_LowerLimit =',data.apply(lambda x:x['Ni_Target']-x['Ni_LowerLimit'],axis=1).mean())

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Histogram(x=data['Cr_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Cr'))
fig.add_trace(go.Histogram(x=data['Ni_exceed'],xbins=dict(start=-0.5, end= 0.5, size=0.005), name='Гистограмма превышения Ni'))
#fig.add_trace(go.Histogram(x=data['Cr_exceed'], name='Гистограмма превышения Cr'))
fig.show()

In [ ]:
data['Cr_adviser_target']=data.apply(lambda x:(x['Cr_Target']+x['Cr_LowerLimit'])/2,axis=1)
data['Ni_adviser_target']=data.apply(lambda x:(x['Ni_Target']+x['Ni_LowerLimit'])/2,axis=1)
data['Cr_exceed']=data.apply(lambda x:x['Cr_Final']-x['Cr_adviser_target'],axis=1)
data['Ni_exceed']=data.apply(lambda x:x['Ni_Final']-x['Ni_adviser_target'],axis=1)

print('в среднем разница между Cr_Target и Cr_LowerLimit =',data.apply(lambda x:x['Cr_Target']-x['Cr_LowerLimit'],axis=1).mean())
print('в среднем разница между Ni_Target и Ni_LowerLimit =',data.apply(lambda x:x['Ni_Target']-x['Ni_LowerLimit'],axis=1).mean())

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Histogram(x=data['Cr_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Cr'))
fig.add_trace(go.Histogram(x=data['Ni_exceed'],xbins=dict(start=-0.5, end= 0.5, size=0.005), name='Гистограмма превышения Ni'))
#fig.add_trace(go.Histogram(x=data['Cr_exceed'], name='Гистограмма превышения Cr'))
fig.show()

### Экономический эффект

In [ ]:
display(data[['TotalIngotsWeight','Cr_Last_EOP','Cr_Final','LFVD_FeCrA','LFVD_FeCrC']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeCrA']==0)&(data['LFVD_FeCrC']==0)&(data['LFVD_FeCrCSi']==0)&(data['LFVD_FeCrC51']==0)]))

In [ ]:
data_cr=data[(data['Cr_Final']>data['Cr_Last_EOP'])&(data['Cr_Final']>data['Cr_Target'])]
data_cr=data_cr.copy()

data_cr['FeCr_1%']=data_cr.apply(lambda x: (x['LFVD_FeCrA']+x['LFVD_FeCrC'])/(x['Cr_Final']-x['Cr_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_cr['Cr_surplus']=data_cr.apply(lambda x:(x['LFVD_FeCrA']+x['LFVD_FeCrC'])-(x['Cr_Target']-x['Cr_Last_EOP'])*x['TotalIngotsWeight']*x['FeCr_1%']/100,axis=1)
display(data_cr[['TotalIngotsWeight','Cr_Last_EOP','Cr_Target','Cr_Final','LFVD_FeCrA','LFVD_FeCrC','FeCr_1%','Cr_surplus']].head(3))
print('перерасход в год FeCr=',round(data_cr['Cr_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_cr['Cr_surplus'].sum()/10*300/1000000,2),'млн.руб')

### Какая стоимость перерасхода если нам удастся обеспечить среднее значение между Cr_LowerLimit и Cr_Target?

### Выполните расчеты для каждого элемента: Mo, Ni, V, W

In [ ]:
import pandas as pd
import dtreeviz
from sklearn.tree import DecisionTreeClassifier

random_state=42
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
#display(data)
data['quality']=data.apply(lambda x: 1 if x['Cr_Final']<=x['Cr_Target'] else 0,axis=1)

X=data.drop(['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).values
y=data['quality'].values

features=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).columns.tolist()
class_names=list(set(y))
target='quality'
tree_classifier = DecisionTreeClassifier(max_depth=33, random_state=random_state)
tree_classifier.fit(X, y)

viz_model = dtreeviz.model(tree_classifier,
                           X_train=X, y_train=y,
                           feature_names=features,
                           target_name=target)

viz_model.view(scale=0.8)
#viz_model.save('dtreeviz.svg')


In [ ]:
importances = tree_classifier.feature_importances_
X=data.drop(['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1)
threshold = 0.1  # Adjust as needed
selected_features = X.columns[importances > threshold]
df=pd.DataFrame([importances],columns=X.columns)
display(df.transpose().sort_values(by=0, ascending=False))
